# 🚀 Few-Shot Cross-Domain 3D Multi-Organ Segmentation
## Fault-Tolerant Benchmark Runner with Hugging Face Hub Auto-Sync

### 🛡️ Built-in Free-Tier & Disconnection Protections:
1. **Hugging Face Hub Cloud Sync:** Checkpoints (`checkpoint_latest.pth`, `checkpoint_best.pth`) and CSV result tables are automatically pushed to your private Hugging Face repository.
2. **Full-Volume 3D Sliding-Window Evaluation:** Reconstructs full 3D patient CT volumes for clinically standard 13-organ Dice and HD95 metrics.
3. **Seamless Cloud Auto-Resume:** If your session disconnects, it pulls the latest weights from Hugging Face and resumes from the exact epoch.
4. **Complete Artifact Export:** Automatically bundles all metrics, per-case details, and convergence curves into a single downloadable `results_bundle.zip`.

### Step 1: Environment & GPU Verification

In [ ]:
# Install lightweight dependencies
!pip install -q nibabel huggingface_hub scipy tqdm pandas matplotlib

import torch
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ GPU not detected. Please enable GPU in Runtime -> Change runtime type (Colab) or Accelerator -> GPU P100 (Kaggle).")

### Step 2: Configure Hugging Face Cloud Storage for Checkpoints
Paste your **Hugging Face Write Token** in the quotes below (`HF_TOKEN = "..."`).

In [ ]:
import os

# -------------------------------------------------------------------------
# HUGGING FACE CREDENTIALS & PRIVATE CLOUD REPO
# -------------------------------------------------------------------------
HF_USERNAME = "Pronob002"
HF_REPO = f"{HF_USERNAME}/hybrid-swin-unet-checkpoints"

# 👉 Paste your Hugging Face Write Token here:
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"

os.environ["HF_TOKEN"] = HF_TOKEN
print(f"✅ Target Hugging Face Cloud Repo: https://huggingface.co/{HF_REPO}")

### Step 3: Load Workspace Code
Clones or updates the repository to the latest version.

In [ ]:
import os, sys
if not os.path.exists("Hybrid_Swin_UNet"):
    !git clone https://github.com/pronob002/Hybrid_Swin_UNet.git
    %cd Hybrid_Swin_UNet
else:
    %cd Hybrid_Swin_UNet
    !git fetch origin master
    !git reset --hard origin/master

sys.path.append(os.getcwd())
print("Working directory:", os.getcwd())

### Step 4: Run 5-Shot Adaptation Benchmark
Runs the **Proposed Hybrid Swin-UNet** and **3D U-Net Baseline** with full-volume 3D sliding window evaluation.

In [ ]:
# 1. Hybrid Swin-UNet (Proposed)
!python train.py --model hybrid_swin --shots 5 --epochs 50 --eval_cases 10 --seed 42 --hf_repo $HF_REPO --hf_token $HF_TOKEN

# 2. 3D U-Net Baseline
!python train.py --model unet3d --shots 5 --epochs 50 --eval_cases 10 --seed 42 --hf_repo $HF_REPO --hf_token $HF_TOKEN

### Step 5: Multi-Regime Benchmark ($k=1, 3, 5, 10$ Shots & Multi-Seed Loop)
Runs the full multi-regime suite across $k \in \{1, 3, 5, 10\}$ shots.

In [ ]:
from scripts.train_fewshot import run_experiment

regimes = [1, 3, 5, 10]
models = ["hybrid_swin", "unet3d"]
seeds = [42, 123, 456]

for seed in seeds:
    for m in models:
        for k in regimes:
            print(f"\n{'='*70}")
            print(f">>> Running {m.upper()} | {k}-Shot Adaptation | Seed: {seed}")
            print(f"{'='*70}")
            run_experiment(
                model_type=m,
                k_shots=k,
                num_epochs=50,
                seed=seed,
                eval_cases=10,
                checkpoint_base_dir="checkpoints",
                results_dir="results",
                hf_repo=HF_REPO,
                hf_token=HF_TOKEN,
                force_rerun=False  # Auto-resumes from HF and skips completed runs!
            )

### Step 6: Post-Benchmark Analysis & Publication Table Generation

In [ ]:
# Run statistical tests (Wilcoxon p-values) and generate CBM tables & figures
!python scripts/analyze_results.py

import pandas as pd
csv_path = "results/benchmark_summary.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("\n--- Completed Benchmark Summary ---")
    display(df)

### Step 7: Create 1-Click Downloadable Zip Bundle
Packages `results/`, `checkpoints/`, and `figures/` into `results_bundle.zip` for instant download.

In [ ]:
import shutil, os
from huggingface_hub import HfApi

!zip -r results_bundle.zip results checkpoints figures
print("\n✅ [SUCCESS] Created results_bundle.zip!")

# Also push the complete zip bundle to Hugging Face
if os.path.exists("results_bundle.zip") and os.environ.get("HF_TOKEN"):
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.upload_file(
        path_or_fileobj="results_bundle.zip",
        path_in_repo="results_bundle.zip",
        repo_id=HF_REPO,
        repo_type="model",
        token=os.environ["HF_TOKEN"]
    )
    print(f"✅ [HF-SYNC] Uploaded 'results_bundle.zip' to https://huggingface.co/{HF_REPO}")

### Step 8: Cloud & Local Storage Cleanup (Delete Redundant Checkpoints)
Safely deletes intermediate `checkpoint_latest.pth` files from your **Hugging Face Hub repository** and local disk to prevent free-tier storage quota exhaustion, while preserving all peak evaluation weights (`checkpoint_best.pth`), result CSVs, and `results_bundle.zip`.

In [ ]:
from scripts.cleanup_checkpoints import cleanup_redundant_checkpoints

# 1. Preview which redundant checkpoints will be purged (Dry Run)
cleanup_redundant_checkpoints(
    repo_id=HF_REPO,
    token=HF_TOKEN,
    checkpoint_dir="checkpoints",
    delete_local=True,
    dry_run=True
)

# 2. Execute Cleanup (Frees Hugging Face Cloud & Local Disk Storage)
# All optimal weights (checkpoint_best.pth), CSV summaries, and zip archives are safely retained!
cleanup_redundant_checkpoints(
    repo_id=HF_REPO,
    token=HF_TOKEN,
    checkpoint_dir="checkpoints",
    delete_local=True,
    dry_run=False
)